# Vendee Globe RL — ERA5 Dynamic Wind (Final)
### Observation Space: 30 features | PPO | Training Seasons: 2016-2017 & 2020-2021

**Training seasons**: November 2016 – March 2017 and November 2020 – March 2021  
**Test season** (unseen): November 2024 – March 2025  
**Wind data**: ERA5 reanalysis (u10, v10) with bilinear interpolation via Scipy `RegularGridInterpolator`  
**Observation**: 30 features — boat state, current weather, wind forecast (3 horizons), navigation, VMG  
**Algorithm**: PPO (Stable-Baselines3) with MlpPolicy [256, 256] for both actor and critic  

---


In [1]:
# Google Drive mount to load datasets
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Dependencies installation
%pip install -q gymnasium numpy matplotlib stable-baselines3 shimmy global-land-mask plotly netCDF4 xarray cfgrib eccodes scipy huggingface-hub

In [3]:
# Dependencies import
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from global_land_mask import globe
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import xarray as xr
from pathlib import Path
from huggingface_hub import snapshot_download
import time
import os
import glob
import warnings
import gc
from scipy.interpolate import RegularGridInterpolator, RectBivariateSpline
warnings.filterwarnings('ignore')

# Create output directories if they do not exist
os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./best_model', exist_ok=True)

print("Imports loaded successfully")


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Imports loaded successfully


In [4]:
# MODE SWITCH — set this value before running the notebook
#   'train'  ->  loads seasons 2016-2017 + 2020-2021 (training)
#   'test'   ->  loads season  2024-2025 (unseen, evaluation only)
MODE = 'train'

# Physical constants
EARTH_RADIUS_KM = 6371.0
MAX_WIND_SPEED = 60.0
MAX_BOAT_SPEED = 30.0
MAX_WAYPOINT_DIST = 5000.0
MAX_COAST_DIST = 500.0
MAX_ICE_DIST = 1000.0
MS_TO_KNOTS = 1.94384
SOUTHERN_ICE_LIMIT = -60.0     # latitude boundary for the ice exclusion zone

# Vendee Globe waypoints (lon, lat, capture_radius_km)
# Capture radious are intentionally generous to not force the agent
WAYPOINTS = [
    [-29.0, 10.0, 1000],      # 1. Mid-Atlantic
    [18.0, -42.0, 1000],      # 2. Good Hope
    [115.0, -48.0, 1000],     # 3. Leeuwin
    [-123.39, -48.88, 1000],  # 4. Pacific
    [-67.0, -56.0, 500],      # 5. Cape Horn
    [-29.0, 10.0, 1000],      # 6. Return Atlantic
    [-3.0, 46.0, 100]         # 7. Finish (Les Sables d'Olonne)
]

START_LAT, START_LON = 46.497, -1.783 # Les Sables d'Olonne

# Hugging Face Repository Configuration
HF_REPO_ID = "NicolasCola7/my-era5-dataset"

# Training seasons (used when MODE = 'train')
REGATTA_SEASONS = [
    {
        'start_year': 2016,
        'race_start': np.datetime64('2016-11-06T13:02:00'),
        'label': '2016-2017'
    },
    {
        'start_year': 2020,
        'race_start': np.datetime64('2020-11-08T13:02:00'),
        'label': '2020-2021'
    },
]

# Test season — never seen during training, used for generalization evaluation
TEST_SEASON = {
    'start_year': 2024,
    'race_start': np.datetime64('2024-11-10T13:02:00'),
    'label': '2024-2025'
}

print(f"Configuration loaded: {len(WAYPOINTS)} waypoints")
print(f"  Start: {START_LAT}N, {START_LON}E")
print(f"  MODE: {MODE}")
if MODE == 'train':
    print(f"  Training seasons: {[s['label'] for s in REGATTA_SEASONS]}")
else:
    print(f"  Test season: {TEST_SEASON['label']}")


Configuration loaded: 7 waypoints
  Start: 46.497N, -1.783E
  MODE: train
  Training seasons: ['2016-2017', '2020-2021']


In [5]:
# Pure mathematical helpers for geographic calculations and wind/angle encoding.

def haversine_distance(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in kilometres between two geographic points."""
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return EARTH_RADIUS_KM * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def haversine_nm(lat1, lon1, lat2, lon2):
    """Compute great-circle distance in nautical miles between two geographic points."""
    return haversine_distance(lat1, lon1, lat2, lon2) / 1.852

def calculate_bearing(lat1, lon1, lat2, lon2):
    """Compute the initial bearing in degrees [0, 360) from point 1 to point 2."""
    lat1_r, lon1_r = np.radians(lat1), np.radians(lon1)
    lat2_r, lon2_r = np.radians(lat2), np.radians(lon2)
    d_lon = lon2_r - lon1_r
    x = np.sin(d_lon) * np.cos(lat2_r)
    y = np.cos(lat1_r) * np.sin(lat2_r) - np.sin(lat1_r) * np.cos(lat2_r) * np.cos(d_lon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

def angle_diff(a, b):
    """Compute the signed angular difference (a - b) in the range [-180, 180]."""
    diff = (a - b) % 360
    return diff - 360 if diff > 180 else diff

def encode_angle(angle_deg):
    """Encode an angle as a (sin, cos) pair to avoid discontinuities at 0/360."""
    rad = np.radians(angle_deg)
    return np.sin(rad), np.cos(rad)

def wind_params(u10, v10):
    """Convert ERA5 wind components (m/s) to True Wind Speed (knots) and True Wind Direction (degrees)."""
    tws = np.sqrt(u10**2 + v10**2) * MS_TO_KNOTS
    twd = (np.degrees(np.arctan2(-u10, -v10)) + 360) % 360
    return tws, twd


In [6]:
class ERA5WindManager:
    """
    Manages ERA5 wind data for a single regatta season (November to March).
    Loads .grib or .nc files, filters to u10/v10 only, normalises longitude
    to [-180, 180] and stores data as float16 arrays to minimise RAM usage.
    """

    def __init__(self, data_path, start_year):
        self.data_path = Path(data_path)
        self.start_year = start_year
        self.times = None
        self.lats = None
        self.lons = None
        self.u_data = None
        self.v_data = None
        self._load_regatta_season()

    def _load_regatta_season(self):
        print(f"Initialising ERA5WindManager at: {self.data_path}")

        # Build the list of expected monthly files (November through March)
        expected_months = [
          (self.start_year, 11),
          (self.start_year, 12),
          (self.start_year + 1, 1),
          (self.start_year + 1, 2),
          (self.start_year + 1, 3)
        ]

        target_patterns = [f"era5_full_{y}_{m:02d}" for y, m in expected_months]
        print(f"  Target season patterns: {target_patterns}")

        # Match file patterns against available files in the data directory
        all_files = sorted(list(self.data_path.glob('*.nc')) + list(self.data_path.glob('*.grib')))
        files_to_load = []

        for pattern in target_patterns:
            match = next((f for f in all_files if pattern in f.name), None)
            if match:
                files_to_load.append(match)
            else:
                print(f"  WARNING: No file found for pattern {pattern}")

        if not files_to_load:
            raise ValueError(f"No files found for season {self.start_year}-{self.start_year+1}")

        print(f"Loading {len(files_to_load)} files: {[f.name for f in files_to_load]}")

        # Load with cfgrib/xarray, filtering to wind variables only
        try:
            engine = 'cfgrib' if files_to_load[0].suffix == '.grib' else None

            ds = xr.open_mfdataset(
                files_to_load,
                combine='nested',
                concat_dim='time',
                parallel=False,              # Dask is unstable on Windows
                engine=engine,
                backend_kwargs={
                    'indexpath': '',         # never generate .idx cache files
                    'filter_by_keys': {'shortName': ['10u', '10v']}  # wind only
                } if engine == 'cfgrib' else {},
                preprocess=lambda d: d.drop_vars(
                    ['mwd', 'tp', 'msl', 'swh', 'mtd', 'mcc'], errors='ignore'
                )
            )

            # Standardise coordinate names
            if 'latitude' in ds.coords:
                ds = ds.rename({'latitude': 'lat', 'longitude': 'lon'})

            # Identify u10 and v10 variable names (may differ between GRIB and NetCDF)
            u_var = 'u10' if 'u10' in ds else '10u' if '10u' in ds else \
                [v for v in ds.data_vars if 'u' in v.lower()][0]
            v_var = 'v10' if 'v10' in ds else '10v' if '10v' in ds else \
                [v for v in ds.data_vars if 'v' in v.lower()][0]

            print("Converting to NumPy arrays (this may take a moment)...")

            self.lats = ds.lat.values.astype(np.float32)
            self.lons = ds.lon.values.astype(np.float32)
            self.times = ds.time.values

            # Normalise longitude from [0, 360] to [-180, 180] if necessary
            if self.lons.max() > 180:
                self.lons = np.where(self.lons > 180, self.lons - 360, self.lons)

            idx = np.argsort(self.lons)
            self.lons = self.lons[idx]
            ds = ds.isel(lon=idx)

            # Store as float16 to reduce memory footprint
            self.u_data = ds[u_var].values.astype(np.float16)
            self.v_data = ds[v_var].values.astype(np.float16)

            ds.close()
            gc.collect()

            ram_gb = (self.u_data.nbytes + self.v_data.nbytes) / 1e9
            print(f"  Data ready: shape={self.u_data.shape} | RAM usage: {ram_gb:.2f} GB")
            print(f"  Period: {self.times[0]} -> {self.times[-1]}")

        except Exception as e:
            print(f"Critical error during data loading: {e}")
            raise e

    def get_time_range(self):
        """Return the (start, end) timestamps of the loaded season."""
        return self.times[0], self.times[-1]

    def get_random_start_time(self, margin_hours=24*2):
        """Return a random start time within the season, leaving a safety margin at the end."""
        t_start, t_end = self.get_time_range()
        ts_start = t_start.astype('datetime64[s]').astype(float)
        ts_end = t_end.astype('datetime64[s]').astype(float)
        margin_sec = margin_hours * 3600

        if (ts_end - margin_sec) <= ts_start:
            return t_start

        random_sec = np.random.uniform(0, (ts_end - ts_start) - margin_sec)
        return t_start + np.timedelta64(int(random_sec), 's')

    def get_wind(self, lat, lon, time):
        """
        Return interpolated (u10, v10) wind components at the given position and time.
        Uses a Scipy RegularGridInterpolator (trilinear) built lazily on the first call.
        """
        if not hasattr(self, '_interp_u'):
            if self.lats[0] > self.lats[-1]:
                grid_lats = self.lats[::-1]
                data_u = self.u_data[:, ::-1, :]
                data_v = self.v_data[:, ::-1, :]
            else:
                grid_lats = self.lats
                data_u = self.u_data
                data_v = self.v_data

            grid_time = self.times.astype('datetime64[s]').astype(float)

            self._interp_u = RegularGridInterpolator(
                (grid_time, grid_lats, self.lons),
                data_u,
                bounds_error=False,
                fill_value=None
            )
            self._interp_v = RegularGridInterpolator(
                (grid_time, grid_lats, self.lons),
                data_v,
                bounds_error=False,
                fill_value=None
            )

        if lon > 180:
            lon -= 360

        t_val = time.astype('datetime64[s]').astype(float)
        u_val = self._interp_u((t_val, lat, lon))
        v_val = self._interp_v((t_val, lat, lon))

        return float(u_val), float(v_val)

    def get_forecast(self, lat, lon, current_time, horizons=[6, 12, 24]):
        """
        Return a wind forecast dictionary for future time horizons (hours ahead).
        """
        forecast = {}
        for h in horizons:
            future_time = current_time + np.timedelta64(h, 'h')
            if future_time > self.times[-1]:
                u, v = self.get_wind(lat, lon, current_time)
            else:
                u, v = self.get_wind(lat, lon, future_time)
            tws = np.sqrt(u**2 + v**2) * MS_TO_KNOTS
            twd = (np.degrees(np.arctan2(-u, -v)) + 360) % 360
            forecast[h] = {'tws': tws, 'twd': twd}
        return forecast

In [7]:
wind_managers = {}

if MODE == 'train':
    seasons_to_load = REGATTA_SEASONS
    print(f"MODE = train -> loading {len(seasons_to_load)} seasons")
else:
    seasons_to_load = [TEST_SEASON]
    print(f"MODE = test -> loading unseen season {TEST_SEASON['label']}")

# 1. Dynamically download only the required months based on the mode
print("\nFetching required files from Hugging Face...")
hf_patterns = []
for season in seasons_to_load:
    y1 = season['start_year']
    y2 = y1 + 1
    hf_patterns.extend([
        f"*era5_full_{y1}_11*",
        f"*era5_full_{y1}_12*",
        f"*era5_full_{y2}_01*",
        f"*era5_full_{y2}_02*",
        f"*era5_full_{y2}_03*"
    ])

# If your dataset is private, add token="hf_YOUR_TOKEN" inside snapshot_download
data_path = snapshot_download(
    repo_id=HF_REPO_ID,
    repo_type="dataset",
    allow_patterns=hf_patterns
)
print(f"Data cached at: {data_path}")

# 2. Instantiate the managers using the high-speed local cache directory
for season in seasons_to_load:
    label = season['label']
    print(f"\n{'='*50}")
    print(f"Loading season {label}...")

    wm = ERA5WindManager(data_path=data_path, start_year=season['start_year'])

    wind_managers[label] = {
        'manager':    wm,
        'race_start': season['race_start'],
        'label':      label
    }

    # Quick sanity check at the race start position
    u, v = wm.get_wind(START_LAT, START_LON, season['race_start'])
    # Using your existing logic to match your snippet:
    tws = np.sqrt(u**2 + v**2) * MS_TO_KNOTS
    twd = (np.degrees(np.arctan2(-u, -v)) + 360) % 360
    print(f"  Sanity check OK: TWS={tws:.1f} kt, TWD={twd:.0f} deg @ {str(season['race_start'])[:10]}")

print(f"\n{len(wind_managers)} season(s) loaded")
print(f"  Keys: {list(wind_managers.keys())}")

MODE = train -> loading 2 seasons

Fetching required files from Hugging Face...


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

✓ Data cached at: /root/.cache/huggingface/hub/datasets--NicolasCola7--my-era5-dataset/snapshots/84fd9d739904bf4f0162e37e5100f67c87670238

Loading season 2016-2017...
Initialising ERA5WindManager at: /root/.cache/huggingface/hub/datasets--NicolasCola7--my-era5-dataset/snapshots/84fd9d739904bf4f0162e37e5100f67c87670238
  Target season patterns: ['era5_full_2016_11', 'era5_full_2016_12', 'era5_full_2017_01', 'era5_full_2017_02', 'era5_full_2017_03']
Loading 5 files: ['era5_full_2016_11.grib', 'era5_full_2016_12.grib', 'era5_full_2017_01.grib', 'era5_full_2017_02.grib', 'era5_full_2017_03.grib']
Converting to NumPy arrays (this may take a moment)...
  Data ready: shape=(604, 721, 1440) | RAM usage: 2.51 GB
  Period: 2016-11-01T00:00:00.000000000 -> 2017-03-31T18:00:00.000000000
  Sanity check OK: TWS=7.9 kt, TWD=343 deg @ 2016-11-06

Loading season 2020-2021...
Initialising ERA5WindManager at: /root/.cache/huggingface/hub/datasets--NicolasCola7--my-era5-dataset/snapshots/84fd9d739904bf4f0

In [9]:
class BoatPhysics:
    """
    IMOCA 60 simplified polar model with 2D bicubic spline interpolation.
    Taken from https://8bitbyte.ca/sailnavsim/wind_response/imoca_60.html
    """

    def __init__(self, max_speed=35.0):
        self.polar_data = self._create_polar_table()
        self._create_interpolator()
        self.max_speed = max_speed

    def _create_polar_table(self):
        """Build the polar lookup table: boat speed (knots) indexed by TWA and TWS."""

        # True Wind Angles (degrees from bow; 0 = head-to-wind)
        wind_angles = np.array([
            30, 40, 50, 60, 70, 80, 90, 100, 110, 120,
            130, 140, 150, 160, 170, 180
        ], dtype=float)

        # True Wind Speeds (knots)
        wind_speeds = np.array([5, 10, 15, 20, 25, 30, 35], dtype=float)

        # Boat speeds matrix [TWA rows x TWS columns]
        boat_speeds = np.array([
            # 5kt   10kt   15kt   20kt   25kt   30kt   35kt
            [4.93,  7.88,  7.44,  7.27,  6.80,  6.73,  6.85],  # 30
            [6.68,  9.78,  9.63,  9.78,  9.52,  9.51,  9.72],  # 40
            [7.76, 10.74, 10.67, 10.95, 10.80, 10.92, 11.22],  # 50
            [8.48, 11.40, 11.52, 11.95, 11.93, 12.17, 12.56],  # 60
            [9.01, 11.99, 12.42, 13.16, 13.42, 13.85, 14.36],  # 70
            [9.27, 12.58, 13.55, 14.72, 15.34, 16.03, 16.70],  # 80
            [9.33, 13.12, 14.91, 16.67, 17.77, 18.72, 19.57],  # 90
            [9.06, 13.54, 16.19, 18.32, 19.62, 20.65, 21.57],  # 100
            [9.11, 13.66, 17.45, 20.04, 21.56, 22.67, 23.68],  # 110
            [9.03, 13.95, 18.76, 21.88, 23.72, 24.88, 25.98],  # 120
            [8.62, 14.04, 18.64, 22.51, 25.17, 26.23, 27.35],  # 130
            [7.84, 13.91, 19.28, 22.95, 25.24, 26.39, 27.54],  # 140
            [6.94, 13.38, 19.41, 23.27, 25.65, 26.82, 27.98],  # 150
            [5.93, 12.31, 17.56, 21.20, 23.52, 24.54, 25.60],  # 160
            [5.28, 11.40, 16.01, 19.23, 21.30, 22.24, 23.20],  # 170
            [4.43,  9.80, 14.38, 17.25, 19.01, 19.88, 20.74],  # 180
        ], dtype=float)

        return {
            'wind_angles': wind_angles,
            'wind_speeds': wind_speeds,
            'boat_speeds': boat_speeds
        }

    def _create_interpolator(self):
        """
        Build a 2D bicubic spline interpolator (RectBivariateSpline).
        Axes: TWA (degrees) x TWS (knots) -> boat speed (knots).
        """
        wind_angles = self.polar_data['wind_angles']
        wind_speeds = self.polar_data['wind_speeds']
        boat_speeds = self.polar_data['boat_speeds']

        self.interpolator = RectBivariateSpline(
            wind_angles,
            wind_speeds,
            boat_speeds,
            kx=3,  # cubic spline on TWA axis
            ky=3   # cubic spline on TWS axis
        )

    def polar_speed(self, twa_deg, tws_knots):
        """
        Return theoretical boat speed (knots) for the given wind conditions.
        The polar is symmetric: TWA is folded to [0, 180] before lookup.
        """
        # Fold TWA to [0, 180] (polar symmetry)
        if twa_deg > 180:
            twa_deg = 360 - twa_deg

        # Clamp inputs to the valid interpolation domain
        twa_deg = np.clip(twa_deg, self.polar_data['wind_angles'][0], self.polar_data['wind_angles'][-1])
        tws_knots = np.clip(tws_knots, self.polar_data['wind_speeds'][0], self.polar_data['wind_speeds'][-1])

        boat_speed_knots = float(self.interpolator(twa_deg, tws_knots)[0, 0])
        return max(0.0, boat_speed_knots)

    def move(self, lat, lon, speed_kt, heading_deg, dt_hours=1.0):
        """
        Advance the boat position on a spherical Earth.
        Returns (new_lat, new_lon) after travelling at speed_kt for dt_hours
        along heading_deg. Longitude is wrapped to [-180, 180].
        """
        dist_km = speed_kt * dt_hours * 1.852
        heading_rad = np.radians(heading_deg)

        d_lat = (dist_km * np.cos(heading_rad)) / 111.32
        new_lat = np.clip(lat + d_lat, -85, 85)

        safe_lat = np.clip(lat, -89, 89)
        d_lon = (dist_km * np.sin(heading_rad)) / (111.32 * np.cos(np.radians(safe_lat)))

        new_lon = lon + d_lon
        while new_lon > 180:  new_lon -= 360
        while new_lon < -180: new_lon += 360

        return new_lat, new_lon


In [10]:
class VendeeGlobeEnv(gym.Env):
    """
    Vendee Globe sailing race environment with real ERA5 wind data.
    """

    def __init__(self, wind_managers=None, max_steps=5000, dt_hours=1.0):
        super().__init__()

        self.max_steps = max_steps
        self.dt_hours = dt_hours
        self.physics = BoatPhysics()
        self.wind_managers = wind_managers
        self.forecast_horizons = [6, 12, 24]  # hours ahead for wind forecast features

        # 30 features, all normalised to [-1, 1]
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(30,), dtype=np.float32)

        # Continuous rudder action: [-1, 1] -> heading delta [-20, +20] degrees
        self.action_space = spaces.Box(
            low=np.array([-1.0], dtype=np.float32),
            high=np.array([1.0], dtype=np.float32),
            dtype=np.float32
        )

        self._reset_state()

    def _reset_state(self):
        """Reset all mutable episode state variables to their initial values."""
        self.boat_lat = START_LAT
        self.boat_lon = START_LON
        self.heading = 225.0
        self.speed = 0.0
        self.rot = 0.0
        self.wp_idx = 0
        self.steps = 0
        self.total_dist = 0.0
        self.sim_time = None
        self.coast_dist = 500.0
        self.forecast = None
        self.wind = (0.0, 0.0)  # (u10, v10) populated by step()
        self._update_leg_dist()

    def _update_leg_dist(self):
        """Recompute the distance to the current target waypoint."""
        t_lon, t_lat, _ = WAYPOINTS[min(self.wp_idx, len(WAYPOINTS)-1)]
        self.leg_dist = haversine_nm(self.boat_lat, self.boat_lon, t_lat, t_lon)

    def reset(self, seed=None, options=None):
        """Reset the environment and randomly select a training season."""
        super().reset(seed=seed)
        self._reset_state()
        self.heading = self.np_random.uniform(195, 220)

        # Randomly sample one of the available seasons for this episode
        season_key = self.np_random.choice(list(self.wind_managers.keys()))
        self.active_season = self.wind_managers[season_key]
        self.wind_manager = self.active_season['manager']
        self.sim_time = self.active_season['race_start']
        return self._get_obs(), {}


    def _get_obs(self):
        """Build and return the 30-dimensional observation vector."""
        obs = np.zeros(30, dtype=np.float32)

        wp_idx = min(self.wp_idx, len(WAYPOINTS)-1)
        t_lon, t_lat, _ = WAYPOINTS[wp_idx]

        u10, v10 = self.wind
        tws, twd = wind_params(u10, v10)
        twa = angle_diff(twd, self.heading)

        # Navigation geometry
        wp_dist    = haversine_nm(self.boat_lat, self.boat_lon, t_lat, t_lon)
        wp_bearing = calculate_bearing(self.boat_lat, self.boat_lon, t_lat, t_lon)
        beta_rel   = angle_diff(wp_bearing, self.heading)

        # Safety distances
        if self.steps % 5 == 0:
            self.coast_dist = self._estimate_coast_dist()
        coast_dist = self.coast_dist
        ice_dist = max(0, abs(self.boat_lat - SOUTHERN_ICE_LIMIT) * 60)  # approx nm

        # Performance metrics
        vmg_wp = self.speed * np.cos(np.radians(beta_rel))  # velocity made good towards waypoint

        # Wind forecast — recomputed every 3 steps
        if self.steps % 3 == 0 or self.forecast is None:
            self.forecast = self.wind_manager.get_forecast(
                self.boat_lat, self.boat_lon, self.sim_time, self.forecast_horizons
            )
        forecast = self.forecast

        # Boat state [0-6]
        obs[0] = self.boat_lat / 90.0
        obs[1] = self.boat_lon / 180.0
        h_sin, h_cos = encode_angle(self.heading)
        obs[2] = h_sin
        obs[3] = h_cos
        obs[4] = self.speed / MAX_BOAT_SPEED
        obs[5] = np.clip(self.rot / 20.0, -1, 1)  # normalised rudder angle
        obs[6] = self.wp_idx / len(WAYPOINTS)

        # Current weather [7-12]
        obs[7]  = tws / MAX_WIND_SPEED
        twd_s, twd_c = encode_angle(twd)
        obs[8]  = twd_s
        obs[9]  = twd_c
        obs[10] = twa / 180.0
        obs[11] = np.clip(u10 / 40.0, -1, 1)
        obs[12] = np.clip(v10 / 40.0, -1, 1)

        # Wind forecast [13-24]: 3 horizons x (tws, twd_sin, twd_cos, delta_tws)
        for i, h in enumerate(self.forecast_horizons):
            f = forecast.get(h, {'tws': tws, 'twd': twd})
            base = 13 + i * 4
            obs[base] = f['tws'] / MAX_WIND_SPEED
            f_sin, f_cos  = encode_angle(f['twd'])
            obs[base + 1] = f_sin
            obs[base + 2] = f_cos
            obs[base + 3] = np.clip((f['tws'] - tws) / 20.0, -1, 1)  # wind speed change

        # Navigation [25-28]
        b_sin, b_cos = encode_angle(wp_bearing)
        obs[25] = b_sin
        obs[26] = b_cos
        obs[27] = min(coast_dist, MAX_COAST_DIST) / MAX_COAST_DIST
        obs[28] = min(ice_dist, MAX_ICE_DIST)   / MAX_ICE_DIST

        # Performance [29]
        obs[29] = vmg_wp / MAX_BOAT_SPEED

        return obs.clip(-1, 1)

    def _estimate_coast_dist(self):
        """
        Estimate the distance (nm) to the nearest land cell by sampling a
        4x4 grid of probe points within a 1.5-degree radius around the boat.
        """
        min_dist = 500.0
        for d_lat in np.linspace(-1.5, 1.5, 4):
            for d_lon in np.linspace(-1.5, 1.5, 4):
                test_lat = self.boat_lat + d_lat
                test_lon = ((self.boat_lon + d_lon) + 180) % 360 - 180
                if globe.is_land(test_lat, test_lon):
                    dist = haversine_nm(self.boat_lat, self.boat_lon, test_lat, test_lon)
                    min_dist = min(min_dist, dist)
        return min_dist

    def step(self, action):
        """
        Execute one simulation step of dt_hours (1 hour).
        Returns (obs, reward, terminated, truncated, info).
        A land or ice collision immediately terminates the episode with -500 reward.
        """
        self.steps += 1
        prev_lat, prev_lon = self.boat_lat, self.boat_lon

        # Apply rudder: map [-1, 1] to a heading change of [-20, +20] degrees
        rudder = action[0] * 20.0
        self.rot = rudder
        self.heading = (self.heading + rudder) % 360

        # Query ERA5 wind at the current position and time
        u10, v10 = self.wind_manager.get_wind(self.boat_lat, self.boat_lon, self.sim_time)
        self.wind = (u10, v10)
        tws, twd = wind_params(u10, v10)
        twa = angle_diff(twd, self.heading)

        # Compute boat speed from the polar model; 0.85 accounts for real-world inefficiencies
        self.speed = self.physics.polar_speed(abs(twa), tws) * 0.85

        # Move the boat on the spherical Earth
        new_lat, new_lon = self.physics.move(
            self.boat_lat, self.boat_lon,
            self.speed, self.heading, self.dt_hours
        )

        # Terminate immediately on land collision or ice zone violation
        if globe.is_land(new_lat, new_lon) or new_lat < SOUTHERN_ICE_LIMIT:
            return self._get_obs(), -500.0, True, False, {'collision': True}

        # Update boat position and simulation clock
        self.total_dist += haversine_nm(prev_lat, prev_lon, new_lat, new_lon)
        self.boat_lat, self.boat_lon = new_lat, new_lon
        self.sim_time = self.sim_time + np.timedelta64(int(self.dt_hours * 3600), 's')

        # Compute reward
        reward = self._compute_reward(prev_lat, prev_lon, action[0], twa, tws)

        # Check waypoint capture
        terminated, wp_reward = self._check_waypoint()
        reward += wp_reward

        truncated = self.steps >= self.max_steps
        return self._get_obs(), reward, terminated, truncated, self._get_info()

    def _compute_reward(self, prev_lat, prev_lon, action, twa, tws):
        """
        Compute the per-step reward signal. The dominant term is geographic
        progress toward the current waypoint; secondary terms penalise
        oscillation, time, and proximity to two known hazard zones.
        """
        wp_idx = min(self.wp_idx, len(WAYPOINTS)-1)
        t_lon, t_lat, _ = WAYPOINTS[wp_idx]

        # 1. Progress reward: distance closed toward the current waypoint (nm * scale)
        dist_prev = haversine_nm(prev_lat, prev_lon, t_lat, t_lon)
        dist_curr = haversine_nm(self.boat_lat, self.boat_lon, t_lat, t_lon)
        progress_reward = (dist_prev - dist_curr) * 2.0

        # 2. Speed bonus
        speed_bonus = self.speed / MAX_BOAT_SPEED * 0.5

        # 3. Stability penalty
        stability_penalty = -abs(action) * 0.3

        # 4. Time penalty: small constant cost per step to incentivise speed
        time_penalty = -0.1

        # 5. Cape Horn coast penalty
        cape_horn_penalty = -3.0 if self.boat_lat < -50 and -75 < self.boat_lon < -55 else 0

        return (progress_reward + speed_bonus + stability_penalty + time_penalty + cape_horn_penalty)

    def _check_waypoint(self):
        """
        Check whether the boat has entered the capture radius of the current waypoint.
        Returns (terminated, bonus_reward). A finish bonus of 5000 is given on race completion.
        Each intermediate waypoint yields a bonus of 1000.
        """
        t_lon, t_lat, radius = WAYPOINTS[self.wp_idx]
        dist = haversine_distance(self.boat_lat, self.boat_lon, t_lat, t_lon)

        if dist <= radius:
            print(f"Waypoint {self.wp_idx + 1} reached (step {self.steps})")

            if self.wp_idx >= len(WAYPOINTS) - 1:
                print("FINISH!")
                return True, 5000.0

            self.wp_idx += 1
            self._update_leg_dist()
            return False, 1000.0

        return False, 0.0

    def _get_info(self):
        """Return a diagnostic info dict for logging and evaluation."""
        return {
            'step': self.steps, 'lat': self.boat_lat, 'lon': self.boat_lon,
            'speed': self.speed, 'heading': self.heading, 'wp_idx': self.wp_idx,
            'total_dist': self.total_dist,
            'sim_time': str(self.sim_time)[:19]
        }


# Instantiate and smoke-test the environment
env = VendeeGlobeEnv(wind_managers=wind_managers, max_steps=5000)
obs, _ = env.reset()
print(f"Environment created successfully")
print(f"Observation size: {obs.shape[0]} features")
print(f"Waypoints: {len(WAYPOINTS)}")
print(f"Obs range: [{obs.min():.3f}, {obs.max():.3f}]")

# Run 10 random steps to verify the environment is functional
for i in range(10):
    act = env.action_space.sample()
    obs, rew, term, trunc, info = env.step(act)
    if term or trunc:
        break
print(f"Random step test OK — speed={info['speed']:.1f} kt, lat={info['lat']:.2f}")
print(f"NaN check: {np.isnan(obs).sum()} NaN values in observation")


Environment created successfully
Observation size: 30 features
Waypoints: 7
Obs range: [-1.000, 1.000]
Random step test OK — speed=6.7 kt, lat=45.34
NaN check: 0 NaN values in observation


In [11]:
class TrainingCallback(BaseCallback):
    """
    Custom SB3 callback that prints a training log every log_freq steps,
    persists the best model to disk, and halts training if the mean episode
    reward falls below crash_threshold for patience consecutive log windows.
    """
    def __init__(self, log_freq=25000, crash_threshold=-700, patience=5):
        super().__init__()
        self.log_freq = log_freq
        self.crash_threshold = crash_threshold
        self.patience = patience
        self.crash_count = 0
        self.best_reward = -np.inf
        self.start_time = None
        self.reward_history = []

    def _on_training_start(self):
        self.start_time = time.time()
        print("\n" + "="*60)
        print("TRAINING STARTED")
        print("="*60)

    def _on_step(self):
        if self.n_calls % self.log_freq == 0:
            elapsed = time.time() - self.start_time
            fps = self.n_calls / elapsed

            if self.model.ep_info_buffer:
                rewards = [ep['r'] for ep in self.model.ep_info_buffer]
                mean_reward = np.mean(rewards)
                self.reward_history.append(mean_reward)

                status = "OK" if mean_reward > -300 else "WARN" if mean_reward > -500 else "CRASH"
                print(f"[{status}] {self.n_calls/1000:.0f}K steps | "
                      f"{elapsed/60:.1f} min | "
                      f"{fps:.0f} fps | "
                      f"reward: {mean_reward:.1f} | "
                      f"best: {self.best_reward:.1f}")

                # Save model whenever a new best reward is achieved
                if mean_reward > self.best_reward:
                    self.best_reward = mean_reward
                    self.model.save('./best_model/vendee_best')
                    print(f"New best model saved (reward: {mean_reward:.1f})")

                # Early stopping on persistent performance collapse
                if mean_reward < self.crash_threshold:
                    self.crash_count += 1
                    if self.crash_count >= self.patience:
                        print("\nEARLY STOPPING triggered")
                        return False
                else:
                    self.crash_count = 0
        return True

    def _on_training_end(self):
        elapsed = time.time() - self.start_time
        print("\n" + "="*60)
        print(f"TRAINING COMPLETED in {elapsed/60:.1f} minutes")
        print(f"Best reward achieved: {self.best_reward:.1f}")
        print("="*60)

In [12]:
env = VendeeGlobeEnv(wind_managers=wind_managers, max_steps=5000)

policy_kwargs = dict(
    net_arch=dict(pi=[256, 256], vf=[256, 256])
)

model = PPO(
    "MlpPolicy", env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.995,
    gae_lambda=0.95,
    clip_range=0.15,
    target_kl=0.015,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    policy_kwargs=policy_kwargs,
    verbose=0
)

In [13]:
TIMESTEPS = 15_000_000

print(f"Starting training: {TIMESTEPS/1_000_000:.0f}M timesteps")
print(f"Estimated time: ~{TIMESTEPS/200000 * 60:.0f} minutes")

training_callback = TrainingCallback(log_freq=25000, crash_threshold=-700, patience=5)
checkpoint_callback = CheckpointCallback(
    save_freq=1_000_000,
    save_path='./drive/MyDrive/checkpoints/',
    name_prefix='vendee'
)

model.learn(
    total_timesteps=TIMESTEPS,
    callback=[training_callback, checkpoint_callback],
    progress_bar=True
)

model.save('./vendee_final')

Starting training: 15M timesteps
Estimated time: ~4500 minutes


Output()


TRAINING STARTED


Waypoint 1 reached (step 262)

Waypoint 1 reached (step 453)

Waypoint 1 reached (step 348)

KeyboardInterrupt: 

In [ ]:
# Load model from latest checkpoint file

checkpoint_dir = './drive/MyDrive/checkpoints/'
checkpoints = glob.glob(os.path.join(checkpoint_dir, 'vendee_*.zip'))

if checkpoints:
    def get_steps(path):
        """Extract the step count from the checkpoint filename."""
        try:
            return int(path.split('_')[-2])
        except:
            return 0

    checkpoints.sort(key=get_steps)
    latest_checkpoint = checkpoints[-1]
    latest_steps = get_steps(latest_checkpoint)

    print(f"Found {len(checkpoints)} checkpoint(s)")
    print(f"Latest: {os.path.basename(latest_checkpoint)}")
    print(f"Steps completed: {latest_steps:,}")

    env = VendeeGlobeEnv(wind_managers=wind_managers, max_steps=5000)
    model = PPO.load(latest_checkpoint.replace('.zip', ''), env=env)

In [ ]:
# Load model from custom path
path = "/your/model/path"

# Enter the number of steps you trained the model for
latest_steps = 0

env = VendeeGlobeEnv(wind_managers=wind_managers, max_steps=5000)
model = PPO.load(path.replace('.zip', ''), env=env)

In [ ]:
TOTAL_TIMESTEPS = 20_000_000
REMAINING_TIMESTEPS = TOTAL_TIMESTEPS - latest_steps

if REMAINING_TIMESTEPS > 0:
    print(f"Resuming training for {REMAINING_TIMESTEPS:,} remaining steps...")

    training_callback = TrainingCallback(log_freq=25000, crash_threshold=-700, patience=5)
    checkpoint_callback = CheckpointCallback(
        save_freq=1_000_000,
        save_path='./drive/MyDrive/checkpoints/',
        name_prefix='vendee'
    )

    model.learn(
        total_timesteps=REMAINING_TIMESTEPS,
        callback=[training_callback, checkpoint_callback],
        progress_bar=True,
        reset_num_timesteps=False  # preserve global step counter for logging continuity
    )
    model.save('./vendee_final')
else:
    print(f"Training already complete ({latest_steps:,} >= {TOTAL_TIMESTEPS:,} steps)")

In [ ]:
# Simulations with test season 2024/2025.

# Remeber to change the 'mode' variable from train to test and run again the dataset load cell.
# In order to do that you will probably need to disconnect and delete your runtime since datasets are quite heavy and the runtime RAM is limited to 12 GB.
finished_races_count = 0
for _ in range(50):
    obs, _ = env.reset()
    print(f"Season: {env.active_season['label']}")
    path_lats  = [env.boat_lat]
    path_lons  = [env.boat_lon]
    speeds     = []
    headings   = []
    sim_times  = [str(env.sim_time)[:19]]
    wind_speeds = []

    for step in range(5000):
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, term, trunc, info = env.step(action)
        path_lats.append(env.boat_lat)
        path_lons.append(env.boat_lon)
        speeds.append(env.speed)
        headings.append(env.heading)
        sim_times.append(str(env.sim_time)[:19])

        # Record wind speed at each step for statistics
        u10, v10 = env.wind_manager.get_wind(env.boat_lat, env.boat_lon, env.sim_time)
        tws, _ = wind_params(u10, v10)
        wind_speeds.append(tws)

        if term or trunc:
            print(f"Episode ended at step {step}")
            break

    total_hours = len(speeds) * env.dt_hours
    total_days  = total_hours / 24

    if env.wp_idx == len(WAYPOINTS)-1:
        finished_races_count += 1

    print(f"\nRESULTS:")
    print(f"Waypoints reached: {env.wp_idx}/{len(WAYPOINTS)}")
    print(f"Total distance: {env.total_dist:.0f} nm")
    print(f"Duration: {total_days:.1f} days ({total_hours:.0f} hours)")
    print(f"Average speed: {np.mean(speeds):.1f} kt")
    print(f"Average wind: {np.mean(wind_speeds):.1f} kt")
    print(f"Period: {sim_times[0]} -> {sim_times[-1]}")

    # 3D orthographic globe visualisation
    fig = go.Figure()

    # Boat trajectory
    fig.add_trace(go.Scattergeo(
        lon=path_lons, lat=path_lats, mode='lines',
        line=dict(width=3, color='cyan'), name='Route'
    ))

    # Waypoints
    fig.add_trace(go.Scattergeo(
        lon=[wp[0] for wp in WAYPOINTS],
        lat=[wp[1] for wp in WAYPOINTS],
        mode='markers+text',
        marker=dict(size=10, color='red'),
        text=[f"WP{i+1}" for i in range(len(WAYPOINTS))],
        textposition='top center', name='Waypoints'
    ))

    # Start and end markers
    fig.add_trace(go.Scattergeo(
        lon=[path_lons[0]], lat=[path_lats[0]], mode='markers',
        marker=dict(size=12, color='lime', symbol='star'), name='Start'
    ))
    fig.add_trace(go.Scattergeo(
        lon=[path_lons[-1]], lat=[path_lats[-1]], mode='markers',
        marker=dict(size=12, color='yellow', symbol='diamond'), name='End'
    ))

    # Ice exclusion zone boundary
    fig.add_trace(go.Scattergeo(
        lon=list(range(-180, 181, 10)), lat=[SOUTHERN_ICE_LIMIT]*37,
        mode='lines', line=dict(width=2, color='lightblue', dash='dash'), name='Ice Zone'
    ))

    fig.update_layout(
        title=f'Vendee Globe RL (ERA5) | {env.wp_idx}/{len(WAYPOINTS)} waypoints | {total_days:.0f} days',
        geo=dict(
            projection_type='orthographic',
            showland=True,  landcolor='rgb(40,40,40)',
            showocean=True, oceancolor='rgb(10,40,80)',
            showcoastlines=True, coastlinecolor='rgb(100,100,100)',
            projection_rotation=dict(lon=-20, lat=-20)
        ),
        paper_bgcolor='black', font=dict(color='white'), showlegend=True
    )
    fig.show()


print(f"Finished races: {finished_races_count}")


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Vendee Globe RL — Simulation Statistics (ERA5 Wind)', fontsize=14, fontweight='bold')

# Boat speed over time
axes[0, 0].plot(speeds, color='cyan', alpha=0.7, linewidth=0.5)
axes[0, 0].axhline(np.mean(speeds), color='red', linestyle='--', label=f'Mean: {np.mean(speeds):.1f} kt')
axes[0, 0].set_xlabel('Step'); axes[0, 0].set_ylabel('Speed (kt)')
axes[0, 0].set_title('Boat Speed'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# Heading over time
axes[0, 1].plot(headings, color='orange', alpha=0.7, linewidth=0.5)
axes[0, 1].set_xlabel('Step'); axes[0, 1].set_ylabel('Heading (deg)')
axes[0, 1].set_title('Heading'); axes[0, 1].grid(True, alpha=0.3)

# Latitude profile
axes[0, 2].plot(path_lats, color='green', alpha=0.7)
axes[0, 2].axhline(SOUTHERN_ICE_LIMIT, color='lightblue', linestyle='--', label='Ice Zone')
axes[0, 2].set_xlabel('Step'); axes[0, 2].set_ylabel('Latitude (deg)')
axes[0, 2].set_title('Latitude Profile'); axes[0, 2].legend(); axes[0, 2].grid(True, alpha=0.3)

# Speed distribution
axes[1, 0].hist(speeds, bins=30, color='cyan', alpha=0.7, edgecolor='white')
axes[1, 0].axvline(np.mean(speeds), color='red', linestyle='--')
axes[1, 0].set_xlabel('Speed (kt)'); axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Speed Distribution'); axes[1, 0].grid(True, alpha=0.3)

# ERA5 wind speed along the route
axes[1, 1].plot(wind_speeds, color='yellow', alpha=0.7, linewidth=0.5)
axes[1, 1].axhline(np.mean(wind_speeds), color='red', linestyle='--', label=f'Mean: {np.mean(wind_speeds):.1f} kt')
axes[1, 1].set_xlabel('Step'); axes[1, 1].set_ylabel('TWS (kt)')
axes[1, 1].set_title('ERA5 Wind Speed Along Route'); axes[1, 1].legend(); axes[1, 1].grid(True, alpha=0.3)

# 2D track (lon vs lat)
axes[1, 2].plot(path_lons, path_lats, color='cyan', alpha=0.7, linewidth=0.5)
for i, wp in enumerate(WAYPOINTS):
    axes[1, 2].plot(wp[0], wp[1], 'ro', markersize=6)
    axes[1, 2].annotate(f'WP{i+1}', (wp[0], wp[1]), fontsize=7, color='red')
axes[1, 2].set_xlabel('Longitude'); axes[1, 2].set_ylabel('Latitude')
axes[1, 2].set_title('2D Track'); axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Final summary
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"Waypoints reached: {env.wp_idx}/{len(WAYPOINTS)}")
print(f"Total distance: {env.total_dist:.0f} nm")
print(f"Duration: {total_days:.1f} days")
print(f"Average speed: {np.mean(speeds):.1f} kt")
print(f"Max speed: {np.max(speeds):.1f} kt")
print(f"Average wind: {np.mean(wind_speeds):.1f} kt")
print(f"Max wind: {np.max(wind_speeds):.1f} kt")
print(f"Min latitude: {min(path_lats):.1f} deg")
print(f"Simulation period: {sim_times[0]} -> {sim_times[-1]}")
